In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import json
import re

def clean_description(text):
    """Xóa các dấu gạch ngang và khoảng trắng thừa ở đầu chuỗi"""
    if pd.isna(text) or text == 'nan':
        return ""
    # Thay thế dấu ':' ở cuối (nếu có) để chuỗi ghép lại đẹp hơn
    cleaned = re.sub(r'^[- ]+', '', str(text)).strip()
    if cleaned.endswith(':'):
        cleaned = cleaned[:-1]
    return cleaned

def clean_hs_code(code):
    """Làm sạch mã HS, bỏ dấu chấm, khoảng trắng"""
    if pd.isna(code) or str(code).strip().lower() == 'nan':
        return ""
    code_str = str(code).split('.')[0]
    return code_str.replace(' ', '').replace('.', '').strip()

def process_multi_sheet_hs_data(input_file):
    print(f"Đang khởi tạo nạp dữ liệu từ file: {input_file}...\n")

    xls_dict = pd.read_excel(input_file, sheet_name=None, dtype=str)

    all_records = []
    total_sheets = len(xls_dict)
    print(f"Tìm thấy tổng cộng {total_sheets} tabs. Bắt đầu xử lý...\n")

    for sheet_name, df in xls_dict.items():
        print(f"⏳ Đang xử lý tab: [{sheet_name}]...")

        try:
            df = df.iloc[:, :3]
            df.columns = ['V', 'Ma_hang', 'Mo_ta']
        except Exception as e:
            print(f"  -> Bỏ qua tab [{sheet_name}] vì không đủ 3 cột.")
            continue

        chapter_notes = ""
        current_chapter = ""

        # Bổ sung thêm biến theo dõi cho Level 2
        lvl0_desc = ""
        lvl1_desc = ""
        lvl2_desc = ""

        sheet_record_count = 0

        for index, row in df.iterrows():
            v = str(row['V']).strip() if not pd.isna(row['V']) else ""
            ma_hang = clean_hs_code(row['Ma_hang'])
            mo_ta = str(row['Mo_ta']).strip() if not pd.isna(row['Mo_ta']) else ""

            if not mo_ta and not ma_hang:
                continue

            if mo_ta.lower().startswith("chương"):
                match = re.search(r'\d+', mo_ta)
                current_chapter = match.group().zfill(2) if match else sheet_name.replace('c', '').zfill(2)
                continue

            if mo_ta.lower().startswith("chú giải") or (v == "" and ma_hang == "" and mo_ta):
                chapter_notes += mo_ta + "\n"
                continue

            cleaned_mo_ta = clean_description(mo_ta)
            full_desc = ""

            # --- TỐI ƯU LẠI STATE MACHINE ---

            if v == '0':
                lvl0_desc = cleaned_mo_ta
                lvl1_desc = ""
                lvl2_desc = "" # Reset toàn bộ các cấp dưới

            elif v == '1':
                if ma_hang and (len(ma_hang) >= 8 or ma_hang.endswith('00')):
                    full_desc = f"{lvl0_desc} -> {cleaned_mo_ta}"
                    all_records.append({"chapter": current_chapter, "hs_code": ma_hang.ljust(8, '0'), "full_description": full_desc})
                    sheet_record_count += 1
                else:
                    lvl1_desc = cleaned_mo_ta
                    lvl2_desc = "" # Reset Level 2 vì Level 1 vừa đổi

            elif v == '2':
                # Nếu có mã HS -> Nó là mã cuối
                if ma_hang and len(ma_hang) >= 6:
                    desc_parts = [lvl0_desc, lvl1_desc, cleaned_mo_ta]
                    full_desc = " -> ".join([p for p in desc_parts if p])

                    all_records.append({"chapter": current_chapter, "hs_code": ma_hang.ljust(8, '0'), "full_description": full_desc})
                    sheet_record_count += 1
                # Nếu KHÔNG có mã HS -> Nó là thư mục cha cho Level 3
                else:
                    lvl2_desc = cleaned_mo_ta

            elif v == '3':
                # Level 3 luôn là mã cuối
                if ma_hang and len(ma_hang) >= 6:
                    # Ghép toàn bộ chuỗi từ level 0 đến level 3
                    desc_parts = [lvl0_desc, lvl1_desc, lvl2_desc, cleaned_mo_ta]
                    # Loại bỏ các đoạn trống (trường hợp bị thiếu dữ liệu)
                    full_desc = " -> ".join([p for p in desc_parts if p])

                    all_records.append({"chapter": current_chapter, "hs_code": ma_hang.ljust(8, '0'), "full_description": full_desc})
                    sheet_record_count += 1

        if chapter_notes and current_chapter:
            with open(f"chapter_{current_chapter}_notes.txt", "w", encoding="utf-8") as f:
                f.write(chapter_notes)

        print(f"  -> Hoàn thành tab [{sheet_name}]: Lấy được {sheet_record_count} mã HS.")

    print("\n" + "="*50)
    print(f"🎉 TỔNG KẾT: Đã bóc tách thành công {len(all_records)} mã HS.")

    if len(all_records) > 0:
        result_df = pd.DataFrame(all_records)
        result_df.to_csv("all_hs_codes_cleaned.csv", index=False, encoding="utf-8-sig")
        print("-> Đã tạo Database File : all_hs_codes_cleaned.csv")

        vector_payload = []
        for _, row in result_df.iterrows():
            page_content = f"Chương: {row['chapter']}\nMã HS: {row['hs_code']}\nMô tả chi tiết: {row['full_description']}"
            vector_payload.append({
                "pageContent": page_content,
                "metadata": {"hs_code": row['hs_code'], "chapter": row['chapter']}
            })

        with open("all_vector_payload.json", "w", encoding="utf-8") as f:
            json.dump(vector_payload, f, ensure_ascii=False, indent=2)
        print("-> Đã tạo Vector Payload: all_vector_payload.json")

if __name__ == "__main__":
    file_path = '/content/Ingestion_HS_CODE.xlsx'
    process_multi_sheet_hs_data(file_path)

Đang khởi tạo nạp dữ liệu từ file: /content/Ingestion_HS_CODE.xlsx...

Tìm thấy tổng cộng 9 tabs. Bắt đầu xử lý...

⏳ Đang xử lý tab: [c6]...
  -> Hoàn thành tab [c6]: Lấy được 27 mã HS.
⏳ Đang xử lý tab: [c7]...
  -> Hoàn thành tab [c7]: Lấy được 135 mã HS.
⏳ Đang xử lý tab: [c8]...
  -> Hoàn thành tab [c8]: Lấy được 101 mã HS.
⏳ Đang xử lý tab: [c9]...
  -> Hoàn thành tab [c9]: Lấy được 73 mã HS.
⏳ Đang xử lý tab: [c10]...
  -> Hoàn thành tab [c10]: Lấy được 37 mã HS.
⏳ Đang xử lý tab: [c11]...
  -> Hoàn thành tab [c11]: Lấy được 41 mã HS.
⏳ Đang xử lý tab: [c12]...
  -> Hoàn thành tab [c12]: Lấy được 80 mã HS.
⏳ Đang xử lý tab: [c13]...
  -> Hoàn thành tab [c13]: Lấy được 20 mã HS.
⏳ Đang xử lý tab: [c14]...
  -> Hoàn thành tab [c14]: Lấy được 13 mã HS.

🎉 TỔNG KẾT: Đã bóc tách thành công 527 mã HS.
-> Đã tạo Database File : all_hs_codes_cleaned.csv
-> Đã tạo Vector Payload: all_vector_payload.json


In [ ]:
# Nén các file có tên bắt đầu bằng "all_" và "chapter_" thành 1 file zip
!zip data_hscode.zip all_hs_codes_cleaned.csv all_vector_payload.json chapter_*_notes.txt

# Gọi lệnh tự động tải file zip vừa tạo về máy
from google.colab import files
files.download('data_hscode.zip')

  adding: all_hs_codes_cleaned.csv (deflated 90%)
  adding: all_vector_payload.json (deflated 93%)
  adding: chapter_06_notes.txt (deflated 50%)
  adding: chapter_07_notes.txt (deflated 50%)
  adding: chapter_08_notes.txt (deflated 51%)
  adding: chapter_09_notes.txt (deflated 56%)
  adding: chapter_10_notes.txt (deflated 46%)
  adding: chapter_11_notes.txt (deflated 59%)
  adding: chapter_12_notes.txt (deflated 59%)
  adding: chapter_13_notes.txt (deflated 55%)
  adding: chapter_14_notes.txt (deflated 48%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#UPLOAD QDRANT

In [ ]:
pip install qdrant-client sentence-transformers

In [ ]:
import json
import os
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")                     
COLLECTION_NAME = os.getenv("QDRANT_COLLECTION_NAME", "hs_codes_agriculture_viet")

print("Đã load API Keys an toàn!")

PAYLOAD_FILE = "all_vector_payload.json"

# Khởi tạo Qdrant Client
qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

print("⏳ Đang tải mô hình AI tiếng Việt (Sẽ mất khoảng 1 phút cho lần đầu tiên)...")
# Tải mô hình AI trực tiếp vào RAM của Google Colab
model = SentenceTransformer('keepitreal/vietnamese-sbert')
print("✅ Tải mô hình thành công!")

def get_embedding(text):
    """Sử dụng mô hình cục bộ để nhúng text thành vector"""
    try:
        # Encode và chuyển từ dạng numpy array sang dạng list (chuẩn của Qdrant)
        vector = model.encode(text).tolist()
        return vector
    except Exception as e:
        print(f"❌ Lỗi khi nhúng vector: {e}")
        return None

def setup_qdrant():
    collections = qdrant.get_collections().collections
    exists = any(c.name == COLLECTION_NAME for c in collections)

    if exists:
        print(f"⚠️ Collection '{COLLECTION_NAME}' đã tồn tại. Đang xóa để tạo lại...")
        qdrant.delete_collection(collection_name=COLLECTION_NAME)

    print(f"🚀 Đang tạo Collection '{COLLECTION_NAME}' mới...")
    qdrant.create_collection(
        collection_name=COLLECTION_NAME,
        # Kích thước vector của mô hình vietnamese-sbert là 768
        vectors_config=VectorParams(size=768, distance=Distance.COSINE),
    )
    print("✅ Đã tạo Collection thành công!")

def ingest_data():
    if not os.path.exists(PAYLOAD_FILE):
        print(f"❌ Không tìm thấy file {PAYLOAD_FILE}")
        return

    with open(PAYLOAD_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)

    total_records = len(data)
    print(f"📦 Tìm thấy {total_records} bản ghi. Bắt đầu quá trình Ingestion...\n")

    points = []
    batch_size = 50

    for idx, item in enumerate(data):
        page_content = item.get("pageContent", "")
        metadata = item.get("metadata", {})

        if not page_content:
            continue

        # In tiến trình mỗi 50 record cho đỡ rác màn hình
        if idx % 50 == 0:
            print(f"  ⏳ Đang xử lý dòng [{idx}/{total_records}]...")

        vector = get_embedding(page_content)
        if not vector:
            continue

        points.append(
            PointStruct(
                id=idx,
                vector=vector,
                payload={
                    "pageContent": page_content,
                    "metadata": metadata
                }
            )
        )

        if len(points) >= batch_size or (idx == total_records - 1):
            qdrant.upsert(
                collection_name=COLLECTION_NAME,
                points=points
            )
            points = []

    print("\n🎉 HOÀN THÀNH INGESTION! Dữ liệu đã sẵn sàng trên Qdrant Cloud.")

# ==========================================
# THỰC THI CHƯƠNG TRÌNH
# ==========================================
if __name__ == "__main__":
    setup_qdrant()
    ingest_data()

⏳ Đang tải mô hình AI tiếng Việt (Sẽ mất khoảng 1 phút cho lần đầu tiên)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Tải mô hình thành công!
🚀 Đang tạo Collection 'hs_codes_agriculture_viet' mới...
✅ Đã tạo Collection thành công!
📦 Tìm thấy 527 bản ghi. Bắt đầu quá trình Ingestion...

  ⏳ Đang xử lý dòng [0/527]...
  ⏳ Đang xử lý dòng [50/527]...
  ⏳ Đang xử lý dòng [100/527]...
  ⏳ Đang xử lý dòng [150/527]...
  ⏳ Đang xử lý dòng [200/527]...
  ⏳ Đang xử lý dòng [250/527]...
  ⏳ Đang xử lý dòng [300/527]...
  ⏳ Đang xử lý dòng [350/527]...
  ⏳ Đang xử lý dòng [400/527]...
  ⏳ Đang xử lý dòng [450/527]...
  ⏳ Đang xử lý dòng [500/527]...

🎉 HOÀN THÀNH INGESTION! Dữ liệu đã sẵn sàng trên Qdrant Cloud.
